# Clips - Jaqueta - Cortando os vídeos originais, de acordo com o Score encontrado na Segmentação (MSO-V3)

Autor:  Viviane da Rosa Sommer

Entrega: 14/02/2025

## Objetivo:
Notebook para cortar os vídeos originais em pequenos clips, por faixa de Score.

> Código em paralelo com 4 workers, fique de olho na memória e CPU.

## Importações Necessárias

In [ ]:
!pip install opencv-python-headless
!pip install imageio[ffmpeg] ffmpeg-python tqdm pandas
import os
import pandas as pd
import cv2
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from imageio_ffmpeg import get_ffmpeg_exe
import numpy as np
ffmpeg_path = get_ffmpeg_exe()
print(f"Usando o ffmpeg em: {ffmpeg_path}")

## Declaração de Constantes e Modelos

In [ ]:
csv_folder_path = "Videos-Resultado-Class"
output_folder_path = "Clips-70-80"
os.makedirs(output_folder_path, exist_ok=True)

## Funções Necessárias

In [ ]:
def cut_video(video_path: str, start_time: float, end_time: float, output_path: str) -> None:
    """
    Corta um segmento de um vídeo e salva no caminho especificado.

    Parâmetros:
    - video_path (str): Caminho do vídeo original.
    - start_time (float): Tempo inicial (segundos).
    - end_time (float): Tempo final (segundos).
    - output_path (str): Caminho do arquivo de saída.
    """
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    start_frame = max(0, int(start_time * fps))
    end_frame = min(total_frames, int(end_time * fps))

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    frame_number = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_number > end_frame:
            break
        if start_frame <= frame_number <= end_frame:
            out.write(frame)
        frame_number += 1

    cap.release()
    out.release()

def process_csv(csv_file: str) -> None:
    """
    Processa um CSV para cortar segmentos de vídeo com base na confiança do modelo.

    Parâmetros:
    - csv_file (str): Nome do arquivo CSV.
    """
    try:
        csv_path = os.path.join(csv_folder_path, csv_file)
        df = pd.read_csv(csv_path, usecols=['Time (s)', 'Positive Confidence'])
        df['Smoothed Confidence'] = df['Positive Confidence'].rolling(window=5, min_periods=1).mean()
        filtered_scores = df[(df['Smoothed Confidence'] >= 0.70) & (df['Smoothed Confidence'] < 0.80)]

        if filtered_scores.empty:
            return

        seconds = sorted(filtered_scores['Time (s)'].values)  
        sequences = []
        current_sequence = []

        for second in seconds:
            if not current_sequence or second - current_sequence[-1] <= 1:
                current_sequence.append(second)
            else:
                if len(current_sequence) >= 10:
                    sequences.append((current_sequence[0], current_sequence[-1]))
                current_sequence = [second]

        if current_sequence and len(current_sequence) >= 10:
            sequences.append((current_sequence[0], current_sequence[-1]))

        video_path = os.path.join(csv_folder_path, csv_file.replace("results.csv", "output.mp4"))
        if not os.path.exists(video_path):
            print(f"Vídeo não encontrado: {video_path}. Pulando...")
            return

        processed_seconds = set()
        joined_segments = []
        for start, end in sequences:
            merged = False
            for i, (s, e) in enumerate(joined_segments):
                if start <= e + 2:
                    joined_segments[i] = (s, max(e, end))
                    merged = True
                    break
            if not merged:
                joined_segments.append((start, end))

        for start, end in joined_segments:
            start_int = int(start)
            end_int = int(end)

            if end_int - start_int < 10:
                end_int = start_int + 10

            segment_name = f"{os.path.splitext(csv_file)[0]}_segment_{start_int}_{end_int}.mp4"
            segment_path = os.path.join(output_folder_path, segment_name)
            cut_video(video_path, start_int, end_int, segment_path)
            for s in range(start_int, end_int + 1):
                processed_seconds.add(s)

        for second in seconds:
            second_int = int(second)
            if second_int not in processed_seconds:
                start_time = max(0, second_int - 5)
                end_time = second_int + 5

                if end_time - start_time < 10:
                    end_time = start_time + 10

                segment_name = f"{os.path.splitext(csv_file)[0]}_segment_{start_time}_{end_time}.mp4"
                segment_path = os.path.join(output_folder_path, segment_name)
                cut_video(video_path, start_time, end_time, segment_path)
                for s in range(start_time, end_time + 1):
                    processed_seconds.add(s)

    except Exception as e:
        print(f"Erro ao processar {csv_file}: {e}")

def merge_videos_with_titles_efficient(folder_path: str, output_path: str) -> None:
    """
    Efficiently merges all video files in the specified folder into a single video.

    Parameters:
        folder_path (str): Path to the folder containing the video files.
        output_path (str): Path to save the merged video file.
    """
    try:
        video_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]
        video_files.sort()

        if not video_files:
            print("No videos found in the specified folder.")
            return

        sample_video = cv2.VideoCapture(os.path.join(folder_path, video_files[0]))
        fps = int(sample_video.get(cv2.CAP_PROP_FPS))
        width = int(sample_video.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(sample_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
        sample_video.release()

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        total_frames = 0
        for video_file in video_files:
            video_path = os.path.join(folder_path, video_file)
            cap = cv2.VideoCapture(video_path)
            total_frames += int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            total_frames += fps * 2
            cap.release()

        with tqdm(total=total_frames, desc="Processing Videos", unit="frame") as pbar:
            for video_file in video_files:
                video_path = os.path.join(folder_path, video_file)
                video_name = os.path.splitext(video_file)[0]
                transition_frame = create_transition_frame(video_name, width, height)
                for _ in range(fps * 2):
                    out.write(transition_frame)
                    pbar.update(1)

                cap = cv2.VideoCapture(video_path)
                while True:
                    ret, frame = cap.read()
                    if not ret:
                        break
                    out.write(frame)
                    pbar.update(1)
                cap.release()

        out.release()
        print(f"Final video saved to: {output_path}")

    except Exception as e:
        print(f"Error merging videos: {e}")


def create_transition_frame(text: str, width: int, height: int) -> np.ndarray:
    """
    Creates a black frame with centered white text as a transition screen.

    Parameters:
        text (str): Text to display on the transition screen.
        width (int): Width of the frame.
        height (int): Height of the frame.

    Returns:
        np.ndarray: The generated frame with text.
    """
    max_width = width - 40
    words = text.split()
    lines = []
    current_line = ""

    for word in words:
        if cv2.getTextSize(current_line + " " + word, cv2.FONT_HERSHEY_SIMPLEX, 1, 2)[0][0] <= max_width:
            current_line += " " + word
        else:
            lines.append(current_line)
            current_line = word

    lines.append(current_line)

    frame = np.zeros((height, width, 3), dtype=np.uint8)
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1
    color = (255, 255, 255)
    thickness = 2
    total_text_height = sum([cv2.getTextSize(line, font, font_scale, thickness)[0][1] for line in lines])
    y_offset = (height - total_text_height) // 2

    for line in lines:
        text_size = cv2.getTextSize(line, font, font_scale, thickness)[0]
        text_x = (width - text_size[0]) // 2
        cv2.putText(frame, line, (text_x, y_offset), font, font_scale, color, thickness)
        y_offset += text_size[1]

    return frame

## Gera crops dos vídeos originais

In [ ]:
csv_files = [f for f in os.listdir(csv_folder_path) if f.endswith(".csv")]
with ThreadPoolExecutor(max_workers=4) as executor:
    list(tqdm(executor.map(process_csv, csv_files), total=len(csv_files), desc="Processing CSV files"))

print("Processing completed!")

## Junta crops de uma mesma faixa em um só vídeo

In [ ]:
merge_videos_with_titles_efficient(output_folder_path, output_folder_path + ".mp4")

In [ ]:
!jupyter nbconvert --to html Crop_Score.ipynb